# 03 · Live drowsiness detection

```
webcam → MediaPipe face landmarks → cut out both eyes → eye model (open / closed)
       → eyes closed for 2 seconds → DROWSY alarm
```

- Needs `models/eye_model.pt` from notebook 02
- Run the cells top to bottom. The last cell opens a webcam window: **press Q in that window to stop**.

## 1. Settings

In [1]:
import os, time, urllib.request
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.transforms import v2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision
import sounddevice as sd

MODEL_PATH = "models/eye_model.pt"
LANDMARKER_PATH = "models/face_landmarker.task"
INPUT_SIZE = 128        # must match notebook 02
CROP_SCALE = 1.4        # eye crop size = 1.4 x eye width (matches the training images)
CLOSED_THRESHOLD = 0.5  # eyes count as closed when p(open) is below this
ALARM_SECONDS = 2.0     # eyes closed this long -> alarm (normal blinks are ~0.1-0.4 s)
CAMERA = 0              # 0 = built-in webcam

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 2. Load the trained eye model
Same model and the **same preprocessing** (`to_model_input`) as in notebook 02.

In [2]:
model = models.mobilenet_v3_small()
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device).eval()

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def to_model_input(x):
    # x: float tensor (N, 1, 64, 64), pixels 0-1 -> normalized tensor (N, 3, INPUT_SIZE, INPUT_SIZE)
    x = F.interpolate(x, size=(INPUT_SIZE, INPUT_SIZE), mode="bilinear", align_corners=False)
    x = torch.cat([x, x, x], dim=1)
    return v2.functional.normalize(x, MEAN, STD)

@torch.no_grad()
def predict_open(eyes):
    # eyes: list of 64x64 grayscale crops -> probability of "open" for each
    x = torch.from_numpy(np.stack(eyes)).unsqueeze(1).to(device).float() / 255
    return model(to_model_input(x)).softmax(dim=1)[:, 1].cpu().numpy()

print("Eye model loaded")

Eye model loaded


## 3. MediaPipe face landmarks
MediaPipe Face Landmarker (the newer version of FaceMesh) finds 478 points on the face.
Its model file (~4 MB) is downloaded once into `models/`.

In [3]:
if not os.path.exists(LANDMARKER_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
    urllib.request.urlretrieve(url, LANDMARKER_PATH)

landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=LANDMARKER_PATH),
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
))
print("Face landmarker ready")

Face landmarker ready


## 4. Cut out an eye
We use the two eye-corner landmarks to find the eye's centre, width and tilt.
The crop is rotated so the eye is level (helps when the head is tilted) and scaled to 64×64,
so it looks like the training images.

In [4]:
# landmark numbers of the eye corners, (left corner, right corner) as seen in the image
EYE_CORNERS = [(33, 133), (362, 263)]

def crop_eye(gray, landmarks, corners):
    h, w = gray.shape
    (x1, y1), (x2, y2) = [(landmarks[i].x * w, landmarks[i].y * h) for i in corners]
    center = ((x1 + x2) / 2, (y1 + y2) / 2)
    angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))   # head tilt
    size = np.hypot(x2 - x1, y2 - y1) * CROP_SCALE     # crop size in pixels
    M = cv2.getRotationMatrix2D(center, angle, 64 / size)
    M[:, 2] += (32 - center[0], 32 - center[1])         # put the eye centre in the middle
    return cv2.warpAffine(gray, M, (64, 64)), center, size

## 5. Alarm sound
A short beep that plays in the background, so the video doesn't freeze.

In [5]:
BEEP = (0.5 * np.sin(2 * np.pi * 880 * np.arange(0, 0.4, 1 / 44100))).astype(np.float32)

def beep():
    sd.play(BEEP, 44100)

## 6. Run the live detector
- Green box = eyes open, red box = eyes closed (with a timer)
- Top right: the two eye crops the model actually sees
- Eyes closed for 2 seconds → red border, **DROWSY!** message and beeping
- `p(open)` is smoothed over the last few frames so one bad frame doesn't change the result

The window may open behind VS Code, so check the taskbar. **Press Q in the window to stop.**

In [6]:
cap = cv2.VideoCapture(CAMERA, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
FONT = cv2.FONT_HERSHEY_SIMPLEX

p_open = 1.0        # smoothed probability that the eyes are open
closed_since = None # time when the eyes closed
last_beep = 0
fps, prev_time = 0, time.time()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("Could not read from the webcam. Is another app using it?")
            break
        frame = cv2.flip(frame, 1)  # mirror view
        now = time.time()

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(now * 1000))

        if result.face_landmarks:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            eyes = [crop_eye(gray, result.face_landmarks[0], corners) for corners in EYE_CORNERS]
            p_open = 0.5 * p_open + 0.5 * predict_open([crop for crop, _, _ in eyes]).mean()

            closed = p_open < CLOSED_THRESHOLD
            if not closed:
                closed_since = None
            elif closed_since is None:
                closed_since = now
            closed_for = now - closed_since if closed_since else 0

            color = (0, 0, 255) if closed else (0, 200, 0)
            status = f"EYES CLOSED  {closed_for:.1f}s" if closed else "EYES OPEN"
            for i, (crop, (cx, cy), size) in enumerate(eyes):
                half = int(size / 2)
                cv2.rectangle(frame, (int(cx) - half, int(cy) - half), (int(cx) + half, int(cy) + half), color, 2)
                x0 = frame.shape[1] - 140 * (i + 1)
                frame[10:138, x0:x0 + 128] = cv2.cvtColor(cv2.resize(crop, (128, 128)), cv2.COLOR_GRAY2BGR)
        else:
            closed_since, closed_for = None, 0
            color, status = (0, 200, 255), "NO FACE"

        if closed_for >= ALARM_SECONDS:
            h, w = frame.shape[:2]
            cv2.rectangle(frame, (0, 0), (w - 1, h - 1), (0, 0, 255), 25)
            cv2.putText(frame, "DROWSY! WAKE UP!", (w // 2 - 330, h // 2), FONT, 2.2, (0, 0, 255), 6)
            if now - last_beep > 0.5:
                beep()
                last_beep = now

        fps = 0.9 * fps + 0.1 / max(now - prev_time, 1e-6)
        prev_time = now
        cv2.putText(frame, status, (20, 50), FONT, 1.2, color, 3)
        cv2.putText(frame, f"p(open) {p_open:.2f}   FPS {fps:.0f}", (20, 95), FONT, 0.8, (255, 255, 255), 2)

        cv2.imshow("Driver Drowsiness Detection (press Q to quit)", frame)
        if cv2.waitKey(1) & 0xFF in (ord("q"), ord("Q")):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    sd.stop()

### Tuning (if needed)
- Alarm goes off while your eyes are open → lower `CLOSED_THRESHOLD` (e.g. 0.35) in cell 1
- Closed eyes not detected → raise `CLOSED_THRESHOLD` (e.g. 0.65)
- Alarm too fast / too slow → change `ALARM_SECONDS`

After changing a setting, re-run cell 1 and then cell 6.